# SciGeoGuard-X — Robustness Corrections v8.1

## Corrected IMERG temporal semantics + geospatially aligned CopDEM resampling + cluster bootstrap + final publication figure

This notebook is a **correction and synthesis layer** over the completed live v8 experiment.

It preserves the successful live Sentinel-2 and ESA WorldCover results from v8 and reruns only the two components that required methodological cleanup:

1. **GPM IMERG V07**
   - v8 interpreted the number of precipitation CSV rows per grid cell as temporal completeness.
   - The ERDDAP dataset declares precipitation `0.0` as its missing/fill value, so zero-rain records can be omitted from CSV serialization.
   - v8.1 therefore verifies the source time axis separately and reconstructs the full expected 48 half-hour slots.
   - The precipitation-rate counterfactuals are then computed against the full 24-hour day.

2. **Copernicus DEM GLO-30**
   - v8 used `scipy.ndimage.zoom` for an upsample/downsample round trip.
   - v8.1 replaces this with **geospatially aligned rasterio reprojection** using explicit source and destination affine transforms.
   - Edge pixels are excluded from round-trip error summaries.

### Additional outputs

- region-cluster bootstrap 95% confidence intervals;
- corrected publication summary table;
- corrected real-data counterfactual table;
- separate candidate figures;
- one stitched four-panel final publication figure;
- provenance and correction manifest;
- complete ZIP export.

### Interpretation boundary

This notebook provides **real-data counterfactual replication**, not independent human ground truth.
The independent human-validation stage remains v7.

## 0. Execution

The main notebook uses live public data for the corrected IMERG and CopDEM reruns.

```python
LIVE_DATA = True
```

A normal Colab CPU runtime is sufficient.

The offline validation copy uses fixtures only to test notebook execution. Fixture results must not be used in a manuscript.

In [ ]:
LIVE_DATA = True
SEED = 20261003
BOOTSTRAP_B = 10000

from pathlib import Path
from datetime import datetime, timezone
import os, json, math, random, re, shutil, gzip, base64, hashlib
import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content") if Path("/content").exists() else Path("/mnt/data")
WORK = ROOT / "scigeoguard_v8_1"
WORK.mkdir(parents=True, exist_ok=True)

print("SciGeoGuard-X v8.1")
print("LIVE_DATA =", LIVE_DATA)
print("WORK =", WORK)

# Part I — Load the actual completed v8 outputs

The successful Sentinel-2 and WorldCover experiments are embedded directly from the user's live v8 result bundle.

The old IMERG and CopDEM tables are retained only for before/after comparison.

In [ ]:
V8_SENTINEL_B64 = """H4sIAH2FwGoC/5WWW2/bRhCF3/sr8l7andteRm9xkKQBmqK1nL4SjES7BGTJoKQk7a/v2aUtqbaTpoBhUrP3b8+c4djfDJt1s+x2fTt265u+ud0s+2bY9bftsGy2i37dt6V1N9z2zW5Y9e1itdkv28XmUz+2d/2IHrtm3Azt3fClXyG+x+/9tvuIrs+ErsdusStLLjbj2C927Xr5aWhv++XQrZvP42Z9024/D2O73X/c7obdvvR9aL7tu3XbfdxuVntsd1gv+y/tcri+7sd+vcDOa69vdNgON+v2ejXcHXdx13fjFivU3ay6aWfjttnuut1++8NFv77pVvtx375bY/JGSOyMGH8/Ta96ptyshk99M5eL9v383S/ysi1NpKxXFCiat5fE3l6Z/vb292MbJ1a15mEa1hcUZhRn5uelC9GPRDOihs7xzsyNmXD28iAKCLt7yJpzVk2q7gixJFLiiOEIipRQMlNh9MB6XMZRdPPEohopquYpxhjnKWJ29CSNJchiamLZMmvARhnBFCW7J6MY0Nmb17/N385Uomkz//Dq1ev5/CvQ8gEa09egMYkXaFmfQpvacI6Qm4dpxCu0PNNH0ITPyXPA2e6psauYYf+G03FMgVII2evhWSV5pkgpWcbdxAkbKCQHOHSMVnmk6NhFKLQtpWA1xi6RCUglpZxyhamcTWNCUCVruYYsMRgORmXNxOk5bpeb3a4fl91t+2u/+7MfV916ub2nZ0fJxTOl5yUX2K6YLHKhF7i9Uv7w5uqkLXCAKKZpwhnbC6aZxRk/pnee3YJqE5hTzE0gjzFXyYFZDAl6iB6zIiQBR8ygETMAuxYowjEncoeSAqBUGTJB7pAbcQoQa+WUHZwSRzXNyVMJiSXQA/eYItvUD1SNNUUQZFypHthBRd/Dzo/K46+yc+HKTk7Yvb46adMQ2A8zCt+zk0fs9Dwosi8c2E0P6LGAEQEqFnekH5JtYoX0YujTTQnRQsE5BIklSYU8kVShheBJkMSiQkZVfJI0wg3AGHqEYotK8QrJI9vNYoKyn6P1Cp49dqs/utWq/6v9MH95sKGDysJJjr58pDKYV4beub1k1hbY5m9+PqqMimvEo8r0BWe42qzk5GNjE/XQRC1yeXjck2JjS5JxIVAaDl9iycUlB6gr4ryx5qmaBQE8GF7xjRICB4gN4A256VHrSpkByBU79ABnLLoK0DVYUQZnCPqoK6b/JPV9bkYBpMDLn5CqbSLBoZSHaSgUUoGfuNlUApKfkIo+5SMk4ThkvY9aAMyKDpBPha0URHCe8lZNCjlUHQ9bwjBLBQEsqJJkODr8DjoBc2i0Lgtpwfayk4WYsAkEIwER/pWHJHuO2vtu3e237cXY/T2s/lfRLMQMVaxmIYgJvb+8OLZB7FnjUa0gZjNJMyrEwmkWFi9HvhbJZGtSRq2TSqzUSC8emTlMdo3alg26gDJg9TUrGQdW+A8FaJpssn8vbgZ14UozNlBjqAyEvMSFYFKYYxmbURyL3LJkYaGiv4TuWqqm5ELuAC3Jt6GlIzQ/sa5/JyRuqEJjfgqttMFEcLVH4aZ7aPw0IeGAdoQGNRhVaChnkBZDMzJ9VqA2GgqBSgAcICi2BSNHIkYUCUUxnb414GSl4Co+gxymVUUVLJcaHHLGYjbdQcD9wBBhXlDavZMlUzg+2CbMh6+Y56D9A/ZM/0LACgAA"""
V8_WORLDCOVER_B64 = """H4sIAH2FwGoC/5WVwW4cNwyG73kKw2fBECmSoo6J4UMPzaGL1IciGEy903TQ2V13ZtaI+/SlZMmIa3trG5CB/YcSJfLTr3n4Nh72bnfYDm5Z+/W4uHEddt24df2yDGv313DvlsNxvhm62/H7MC3t110/jdvuj7m/WfMKVb2ZbFq3He+GeRnX+yfysLj90M/Dsv538u/jNOZP3bj/vy/H/fj3sWQ/2nrPPg/f+93tNCwfPg37bxYzH7uf9tuxd5NtyW2+XF5ebTbuavOxuz7M0/byYBvtwO869AjdHXrffQa88pHdrr91DEkJHFx4F935b+DdGdoINsgG2xAb6r+elyB/AcRJE2uof+QYhcvcC++D9x45hZQIyJ0VjVNKLAxINqqmFhdiUPIhVA1sUkpBk5BSnQvRJKHkKSq3uGRzPQfyHKVqiBYXJZBqjFUiiwKJSiLapLwaKAePMTxoIYeFJBGVQaqmecccKTFAXY44Z7WEdjbiqpVTBJYUQp0qOYrAskidKPlUkkiRENKDFnMYQhAMdrKm5Tih4O1flfL6XtQErikVckpUL7bnpuXCSQhRkxf8ev7hl8O6DvO233Wfh/XPYZ76/XZ5DyAMV5alAJIQbLNPAHkOhztLPwCSd8HOKhe1YZGLB0BEPkFtxpPCPGJhkv1mDAlrRcHnsJgJovA6KVgKz9EgsF42Lcd5i4ko0kjJBQSrIEOUE/Q8RwVLcxFVFbSmDbkfYCRjAo9Ny/ioekBJtUchp0gJQzREmpYrEC0VB4GGY245WM8BrAgncCwciKCVJ2jTcvVA1AANULVyIRGQkQiMjcthv8799Gs/TcN992Xz8T1cBLkGfMaFnjSOp2xY6wodMV/7E3C80R9e7vBbvKAs9tjhdxb1BS/gJwC+agZlb2jmZc7WLn7OaucAtTY1gyi4sF2MYBf7VTeIP0LV3KBsLiiQpEauZtI0BVClVhPNy2GKTBI0Ghk/9/v+uHSf5v6fcXoHFRsv12ZHlQpkeaBCfnCLV2jwkVTMzIxOx0SPblFKzmyls9EekeeQQH5YYrLe5ma3ByPHJQ1CyHoCkrfe5Reeh7ciQeXhMw+IkUNtLBfujG+IGqFhkteL+e206VUrxKL5oznEKVBeIiDfeVA7jvU2NQIyilYTe7T0IQeUhzmRcCKsuENZ0JxLgvlNfkv+BZO8iQhACQAA"""
V8_IMERG_B64 = """H4sIAH2FwGoC/5VUTW8bNxC951folgagWA6/eUxctzEQ92AjvRLEinYW2A+Hu2vH+fV9XEt25FQFQkEEOcMh37w3syXftuPAdmnOrB93mU1zmpeJlfFhYrel3cUmd93E+pyG2A5zLvepm+JdLqvjyd6MpeRmjrvUdo8xNc3Sx75nffp20lXDHso43MaSHvCfc5zgGpY+l7Y59sO37l+cuxbbGlhjxvtcKurc52GON6mZx3I4s8alaf/wTYET6cbxJs5lyewO2Nq7FsHV2ozd0g9sGpfS5LgM7TyxQ8rxC6wTy6WM5c2HPNymbilLvBjwDJNC6q0wWylY195ndv357Oz8+prJ4LlgZOvkuZWGkeCSyDhtjCCzWgL3ZARuwDVcS0U6kBCKqo9jERQF7YOhoCyTuEpwEtIq3OKDdsp6TceJsMvL3z8WnDPsBNSwJfME9c/3F5/O/2A/jqtlmNs+n9dUf3t7cXl+9dfm65LL46bkeSlD3m2GcbMXY3P0Mn/77s3VOIOyXerj33n+kkuXht20f1dupXxFkasUIXNQpHhggSPzEHwIQTmNmZHhXmAlvbBmz2MIDmdWvwdLygrlpbZBGXIg9kCTUV7bl9mcpul/QJN4JusA2ugKN1Tg3GtJwrzowaqIWBitpVgtGqYghSCllcHa14PCQ7Y1wsMknTXekHZGIk0nzD4Dezzc6QTOUPsldf+krsuP8fP1+2fGxSvw3lTwNQPUpSOANYe5AtVeEuiU1q12pjmKUwQy3oFpH6xWEkKoNahmK0EAsrRKamjkvdxj178MluinDjLKHTpIKf7UMCgYHcARfODJcYtyIKmcZwrnQtBEwq3NY8l76ILCEB4I98CEPf6dxnmZhrRM8UNJ39tuT6j6qRqUp+cSRk3gZfBkQIdG80J39DCTmmttBHrVYKO4E1Z7KO119VdmqeofyKLWpXbPJYw0fxy/CNZvyb/qt7VoRWWnakfoIe+cqxpjyFVPtJXBR6pqqLB2ePfJLfR/2g53noT2L8kDmvZlBgAA"""
V8_COPDEM_B64 = """H4sIAH2FwGoC/7WTTW/UMBCG7/srqp5NmPG3udHtquKwBbEqPSBkmY27teRNVo5TCr+eybYFWhWxQqoPcTIzfmfmybjETeo7tu3byIYa6jiwmnJk6zKwLtR0E32JQ5/HOoUNY7kK6+in8MxuYqlpHbJv6dz2Ifwm5NT6XbqN+ZdEzPEmTAp+G9sUaHvGk543h1syj7shbHc5dRsfbtPgqYraF9b1dIoKWOeQSNlvCqXesqEfC1X54P3dAPn2hfl1P3bVp+4q32W512tjoeyt78e6G6u/KmSe3N9SvSabb9NQU7eu/j7DvqjICqm1taQdtUfNha/7fNRFKX2hnH/4qZun7tlJ7DYhj2X07zqiwzLVwFYX8/litWLzfhdLl9bj4E9XSz9/f+YR/DlyD+AXYMy0ny6WbPFhdfZGCq7Z8WdoAICbR8uyo2fNX44ZCbPF2ZID2KPXR3shYZViWimhmQPdCDBcgTNaaa6YVabhUkkrhLNSGMWcwoZOKAeWg0CKEQ0wBCaAOXqDxj5ZzJCoUEIBSk4iljlhGmUsWk12pESzj32tsbRh689jvY4lh64dDsKj8A4PyL/ikagfLfm/fOgUe4WmQW6JEXIOZkLABDbagbT0ME5ZIvVvKNBwLgm2xgmJ4CgM47KRoIxzVivDOf2C2Tx2tYT8KeQcv/uL1duDoAg7wbikCl98ZtA1UqNAC6hBUU9kgkahdVIaCdwpiUzIBow0CFYqJw3X+hBAaJDmkIbEghCIag9o4gJaE2WhlDSzZegCdX9Swo+UD4GzupuUS9D44nCmwRBWEhurpeaEY4KjhVOghSRBPcVa12iFIBzy6ZrgQXAEgZRSKC2VpTuEkhnbgEPl6JNgc3Szn6Dldm3zBQAA"""
V8_COUNTERFACTUAL_B64 = """H4sIAH2FwGoC/9Wa21LbOBjH73kKPYCT6mhJl+WwXWYa2iFbeqkRthI0Y8sZH9KyF/sa+0D7YvvZCQ3QFHLoOMEwjDHB+un//fMdDKWb+iJEVdGUiYtKN8t8Yuv2zGYmtbU1eZG6qJi50tbtK+e29DbUUdlkzmT21mVR7mzVlC41bjJxSW2Czd1PF+c2a9zJqQtTOCkbcxlSb6OxC7UPLhtQ9JG+jyimfIAJfL5bnLIBI1Hm5y66Or+5RN98fYdOsUJXl9fIhhTOOYJFoirx7Z0mAJ9l9+1iPo0+X3+6ubhqSYKxt1WRNbUzPqTuu0k9UJUuJM7MK/OtLMLUPOwMD3GsuZaEMhbjmDEl9uAeN7dV7WtYGp0SgsZfgXxSlD92cdvUqAa1a9hH1WQ1shVq9/p8Sz4sNnV98ceXvy7Oo8pPg5lkfmYmpU260AA4oZxxyhVXhAlAIZuRqx/kBB9GcRlrgqmQjGkupeB7cPereMyI4iyW4BhGFaMn10VduzK1ubly9Z0rM1CtWo/PV4aJBwz3LzshmIDJGcNECqa12pu+X/Epl6C/1CSWMeFb8uuVechB1MeaCEHjNtNQrCWme9P3rL5kMcVYSgVvWQmhODmDm5Q2u4GbuHvzZfx+PTtbOUccJN0QxrmgmmnMmMCK7QXer+hYEdBaMyyIFpJStSn7wXM8IZBhNLxlhcA6hpy5F3nPqoPJoUYpjbmIJcbkZGSDbSpzWtq/fXasfYyWQkrCJVFCAUe8M3WvakNBxXFrc0UVJRSzTbDlClsfLJszCfU0hkLEMdc7U/drbQE9royhZZTg7S6LP2+9Lsbv0deizNKzYu7KqKphEkhM212aWVmkTVIv1XYWgOpBcH56dwsTBWpHiWlRtjwtq81hvAjTF4XvvjFJZqtqhUmGeGeqUw9rAhnyAWrrrMi6OQYVExSK3Acga6vsIGnvgrp1XfWaksvvn2OCdbnQSgvFlgf/RUk/Dkn3Y+tXWI1jKtZUjONQcneuflWEEtzp+DQ1HYeGuzH1qh+WXMUw5hH68yOBD59H6HJ0cf0B3WAZPTSJdJnTL4FuWoJaaFa6xM98veDsLv33L8JDge5er0ip9XDdJkmTN4udmjyPCB5SQiB5Q2dFRLw13LjJ0Wj07s8SdRF0VVczCygzsErtZ9k9BBXd3qO0WTwF6spo9/DnoQQBxSuihiZ3YBTTBqKNpsvh1WYC2hZlRNck+JeZRyDHM+gV1HKtB7hONfRYtQ14H8Juiompy8Y9Ub19L2EKs6eQSnOYgxQnv0in67ZBB5T24As9ZITCVAxtN4OZQeudEI/DHbuQH94jgiker76KzbcBow4RPXgEDzkFCwvOKe68zHdifGMmeUR+cJPETw+5pplYb3Pcjz80hyksxozGUgmYJrfmOw5vbEt9cF9sKDQhPTUZaghDrcQCplyxNdob8sAj6oN7AMdPP541yesczPqpG4QMlWCUc8w0xXCwrdiOww/bEB9BwymeHJuYQQ2I6qVIUAINppJSCtwddCu4t+KGFfGB3bBuXjpr/0gffNJU6PxihD58/NQ+UVwOzufj0coHizHZIbiI6gIRjHI0LX2Kvt35zAF2bX1oFf+HtT9a/HMA+AaUCzYk7kWbzPx3l5mkaEBcHyZLkyxl1r+Fe4H1mL7VHqZ1n6MAvzZ3i6uwfFHmC8VekTuFgM1dahb0P8Rf2tCkvqp9SGqz0MIsYg6+V8+OX7SgbyU2v4v+mCP0cxPwVqLzO8iPOTJPE/Bbicq+1Mcckf8BTeSuQR8mAAA="""

def decode_csv(payload):
    from io import BytesIO
    raw = gzip.decompress(base64.b64decode(payload.encode()))
    return pd.read_csv(BytesIO(raw))

sentinel_v8 = decode_csv(V8_SENTINEL_B64)
worldcover_v8 = decode_csv(V8_WORLDCOVER_B64)
imerg_v8_old = decode_csv(V8_IMERG_B64)
copdem_v8_old = decode_csv(V8_COPDEM_B64)
counterfactual_v8_old = decode_csv(V8_COUNTERFACTUAL_B64)

print("Sentinel v8:", sentinel_v8.shape)
print("WorldCover v8:", worldcover_v8.shape)
print("IMERG old:", imerg_v8_old.shape)
print("CopDEM old:", copdem_v8_old.shape)

# Part II — Live dependencies

In [ ]:
if LIVE_DATA:
    import sys, subprocess

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "requests>=2.31",
        "rasterio>=1.3",
        "pyproj>=3.6",
        "affine>=2.4"
    ])

print("Runtime ready.")

# Part III — Regions and source registry

In [ ]:
REGIONS = {
    "Bengaluru_India": {
        "bbox": [77.56, 12.95, 77.62, 13.01],
        "imerg_dates": ["2024-05-20", "2024-09-15"]
    },
    "Rotterdam_Netherlands": {
        "bbox": [4.43, 51.90, 4.51, 51.98],
        "imerg_dates": ["2024-02-22", "2024-10-15"]
    },
    "CentralValley_USA": {
        "bbox": [-121.85, 38.45, -121.77, 38.53],
        "imerg_dates": ["2024-02-05", "2024-11-20"]
    },
    "Manaus_Brazil": {
        "bbox": [-60.08, -3.16, -60.00, -3.08],
        "imerg_dates": ["2024-03-15", "2024-08-18"]
    }
}

SOURCE_REGISTRY = {
    "imerg_v07": {
        "dataset_title":
            "GPM IMERG Final Precipitation L3 Half Hourly 0.1 degree x 0.1 degree V07 2024",
        "dataset_id":"precipitation_2024_v07",
        "erddap":
            "https://erddap.aoml.noaa.gov/hdb/erddap/griddap/precipitation_2024_v07",
        "precipitation_units":"MM/Hr",
        "time_interval_hours":0.5,
        "expected_intervals_per_day":48,
        "lat_grid_start":-89.95,
        "lon_grid_start":-179.95,
        "grid_step_degree":0.1,
        "important_serialization_note":
            "ERDDAP metadata declares precipitation missing_value/_FillValue as 0.0; "
            "zero-rain rows can therefore be absent from CSV output. "
            "Temporal completeness must be verified from the time coordinate, not row counts."
    },
    "copdem": {
        "product":"Copernicus DEM GLO-30",
        "base":"https://copernicus-dem-30m.s3.amazonaws.com",
        "surface_model":"DSM",
        "horizontal_crs":"EPSG:4326",
        "vertical_datum":"EGM2008 / EPSG:3855",
        "nominal_resolution_m":30
    }
}

display(pd.DataFrame([
    {"source":k, **v}
    for k,v in SOURCE_REGISTRY.items()
]))

# Part IV — Corrected IMERG method

## What changes from v8?

The v8 field `mean_intervals_per_cell` counted rows returned for the precipitation variable.

That is **not a valid completeness measure** for this ERDDAP representation because precipitation value `0.0` is marked as missing/fill metadata and may be omitted from CSV output.

v8.1 instead:

1. queries the **time coordinate itself** for the requested UTC day;
2. requires exactly **48 half-hour time coordinates**;
3. constructs all expected 0.1° grid cells in the ROI;
4. treats precipitation rows absent from the CSV as zero-rain slots under this source's serialization convention;
5. computes:
   - correct accumulation: `sum(rate × 0.5 h)`;
   - invalid raw-rate sum labeled as accumulation;
   - invalid full-day mean rate labeled as accumulation.

For any rainy grid cell:

\[
\frac{\sum r_t}{\sum r_t \times 0.5}=2
\]

and

\[
\frac{\frac{1}{48}\sum r_t}{\sum r_t \times 0.5}=\frac{1}{24}.
\]

In [ ]:
def detect_col(df,needle):
    matches = [
        c for c in df.columns
        if needle.lower() in c.lower()
    ]
    if not matches:
        raise KeyError(
            f"No column containing {needle!r}: {list(df.columns)}"
        )
    return matches[0]

def grid_centers_within(lo,hi,start,step=0.1):
    eps=1e-8
    k0=math.ceil((lo-start)/step-eps)
    k1=math.floor((hi-start)/step+eps)

    if k1 < k0:
        # choose nearest center to midpoint
        mid=(lo+hi)/2
        k=round((mid-start)/step)
        return np.array([start+k*step],dtype=float)

    return start + step*np.arange(k0,k1+1,dtype=float)

def imerg_time_url(date):
    base=SOURCE_REGISTRY["imerg_v07"]["erddap"]
    return (
        base +
        ".csv?time"
        f"[({date}T00:00:00Z):1:({date}T23:30:00Z)]"
    )

def imerg_precip_url(date,lat_vals,lon_vals):
    base=SOURCE_REGISTRY["imerg_v07"]["erddap"]

    return (
        base +
        ".csv?precipitation"
        f"[({date}T00:00:00Z):1:({date}T23:30:00Z)]"
        f"[({lat_vals.min():.2f}):1:({lat_vals.max():.2f})]"
        f"[({lon_vals.min():.2f}):1:({lon_vals.max():.2f})]"
    )

def fetch_csv(url,timeout=120):
    import requests
    from io import StringIO

    r=requests.get(url,timeout=timeout)
    r.raise_for_status()
    return pd.read_csv(StringIO(r.text))

def validate_imerg_time_axis(date):
    d=fetch_csv(imerg_time_url(date))

    tcol=detect_col(d,"time")
    times=pd.to_datetime(
        d[tcol],
        errors="coerce",
        utc=True
    ).dropna().drop_duplicates().sort_values()

    expected=SOURCE_REGISTRY["imerg_v07"][
        "expected_intervals_per_day"
    ]

    return {
        "count":int(len(times)),
        "expected":int(expected),
        "valid":bool(len(times)==expected),
        "first":str(times.iloc[0]) if len(times) else None,
        "last":str(times.iloc[-1]) if len(times) else None
    }

def corrected_imerg_one(region,date,bbox):
    lat0=(bbox[1]+bbox[3])/2
    lon0=(bbox[0]+bbox[2])/2

    # Use a compact but multi-cell neighborhood.
    d=0.16

    lat_vals=grid_centers_within(
        lat0-d,
        lat0+d,
        SOURCE_REGISTRY["imerg_v07"]["lat_grid_start"]
    )

    lon_vals=grid_centers_within(
        lon0-d,
        lon0+d,
        SOURCE_REGISTRY["imerg_v07"]["lon_grid_start"]
    )

    time_check=validate_imerg_time_axis(date)

    if not time_check["valid"]:
        raise RuntimeError(
            f"Source time axis has {time_check['count']} "
            f"intervals; expected {time_check['expected']}."
        )

    try:
        raw=fetch_csv(
            imerg_precip_url(
                date,
                lat_vals,
                lon_vals
            )
        )
    except Exception as e:
        # If the server returns no precipitation table for a dry ROI/day,
        # retain an empty table and reconstruct all slots as zero.
        raw=pd.DataFrame()

    expected_intervals=time_check["expected"]
    duration_h=SOURCE_REGISTRY["imerg_v07"]["time_interval_hours"]

    expected_cells=pd.MultiIndex.from_product(
        [
            np.round(lat_vals,2),
            np.round(lon_vals,2)
        ],
        names=["latitude","longitude"]
    ).to_frame(index=False)

    if len(raw):
        pcol=detect_col(raw,"precip")
        latcol=detect_col(raw,"lat")
        loncol=detect_col(raw,"lon")

        raw[pcol]=pd.to_numeric(
            raw[pcol],
            errors="coerce"
        )

        raw[latcol]=pd.to_numeric(
            raw[latcol],
            errors="coerce"
        )

        raw[loncol]=pd.to_numeric(
            raw[loncol],
            errors="coerce"
        )

        clean=raw.dropna(
            subset=[pcol,latcol,loncol]
        ).copy()

        clean["latitude"]=clean[latcol].round(2)
        clean["longitude"]=clean[loncol].round(2)

        cell_sums=(
            clean
            .groupby(["latitude","longitude"])[pcol]
            .agg(
                observed_nonzero_records="count",
                rate_sum="sum"
            )
            .reset_index()
        )
    else:
        pcol="precipitation"
        clean=pd.DataFrame()
        cell_sums=pd.DataFrame(
            columns=[
                "latitude",
                "longitude",
                "observed_nonzero_records",
                "rate_sum"
            ]
        )

    cells=expected_cells.merge(
        cell_sums,
        on=["latitude","longitude"],
        how="left"
    )

    cells["observed_nonzero_records"]=(
        cells["observed_nonzero_records"]
        .fillna(0)
        .astype(int)
    )

    cells["rate_sum"]=(
        pd.to_numeric(
            cells["rate_sum"],
            errors="coerce"
        ).fillna(0.0)
    )

    cells["zero_reconstructed_slots"]=(
        expected_intervals -
        cells["observed_nonzero_records"]
    ).clip(lower=0)

    cells["correct_daily_accum_mm"]=(
        cells["rate_sum"] * duration_h
    )

    # Invalid counterfactual 1:
    cells["wrong_raw_rate_sum_numeric"]=cells["rate_sum"]

    # Invalid counterfactual 2:
    # full-day arithmetic mean rate over all 48 expected half-hour slots.
    cells["wrong_full_day_mean_rate_numeric"]=(
        cells["rate_sum"] / expected_intervals
    )

    rainy=cells[
        cells.correct_daily_accum_mm > 1e-12
    ].copy()

    if len(rainy):
        rainy["raw_sum_overstatement_factor"]=(
            rainy.wrong_raw_rate_sum_numeric /
            rainy.correct_daily_accum_mm
        )

        rainy["mean_as_accum_fraction_of_true"]=(
            rainy.wrong_full_day_mean_rate_numeric /
            rainy.correct_daily_accum_mm
        )

        factor_raw=float(
            rainy.raw_sum_overstatement_factor.median()
        )

        factor_mean=float(
            rainy.mean_as_accum_fraction_of_true.median()
        )
    else:
        factor_raw=np.nan
        factor_mean=np.nan

    expected_records=int(
        len(cells)*expected_intervals
    )

    observed_records=int(
        cells.observed_nonzero_records.sum()
    )

    reconstructed=int(
        cells.zero_reconstructed_slots.sum()
    )

    return {
        "region":region,
        "date":date,
        "mode":"live",
        "status":"SUCCESS",
        "source_time_axis_count":time_check["count"],
        "source_time_axis_expected":time_check["expected"],
        "source_time_axis_complete":time_check["valid"],
        "grid_cells":int(len(cells)),
        "expected_cell_time_records":expected_records,
        "returned_nonzero_or_unmasked_records":observed_records,
        "zero_reconstructed_records":reconstructed,
        "zero_reconstructed_fraction":
            reconstructed/expected_records if expected_records else np.nan,
        "rainy_grid_cells":int(len(rainy)),
        "mean_correct_daily_accum_mm":
            float(cells.correct_daily_accum_mm.mean()),
        "max_correct_daily_accum_mm":
            float(cells.correct_daily_accum_mm.max()),
        "mean_wrong_raw_rate_sum_numeric":
            float(cells.wrong_raw_rate_sum_numeric.mean()),
        "mean_wrong_full_day_mean_rate_numeric":
            float(cells.wrong_full_day_mean_rate_numeric.mean()),
        "median_raw_sum_overstatement_factor":
            factor_raw,
        "median_mean_as_accum_fraction_of_true":
            factor_mean,
        "theoretical_raw_sum_factor":1/duration_h,
        "theoretical_mean_fraction":
            1/(expected_intervals*duration_h),
        "source_units":"MM/Hr",
        "interval_hours":duration_h,
        "serialization_interpretation":
            "Rows absent under ERDDAP 0.0 missing/fill convention are reconstructed as zero-rain slots."
    }

def run_corrected_imerg():
    rows=[]

    if not LIVE_DATA:
        rng=np.random.default_rng(SEED)

        for region,cfg in REGIONS.items():
            for date in cfg["imerg_dates"]:
                cells=16
                rates=np.maximum(
                    0,
                    rng.gamma(
                        shape=1.1,
                        scale=1.2,
                        size=(48,cells)
                    ) - 1.1
                )

                rate_sum=rates.sum(axis=0)
                correct=rate_sum*0.5
                rainy=correct>1e-12

                rows.append({
                    "region":region,
                    "date":date,
                    "mode":"offline_fixture",
                    "status":"SUCCESS",
                    "source_time_axis_count":48,
                    "source_time_axis_expected":48,
                    "source_time_axis_complete":True,
                    "grid_cells":cells,
                    "expected_cell_time_records":48*cells,
                    "returned_nonzero_or_unmasked_records":
                        int((rates>0).sum()),
                    "zero_reconstructed_records":
                        int((rates==0).sum()),
                    "zero_reconstructed_fraction":
                        float((rates==0).mean()),
                    "rainy_grid_cells":int(rainy.sum()),
                    "mean_correct_daily_accum_mm":
                        float(correct.mean()),
                    "max_correct_daily_accum_mm":
                        float(correct.max()),
                    "mean_wrong_raw_rate_sum_numeric":
                        float(rate_sum.mean()),
                    "mean_wrong_full_day_mean_rate_numeric":
                        float((rate_sum/48).mean()),
                    "median_raw_sum_overstatement_factor":
                        float(np.median(rate_sum[rainy]/correct[rainy])),
                    "median_mean_as_accum_fraction_of_true":
                        float(np.median((rate_sum[rainy]/48)/correct[rainy])),
                    "theoretical_raw_sum_factor":2.0,
                    "theoretical_mean_fraction":1/24,
                    "source_units":"MM/Hr",
                    "interval_hours":0.5,
                    "serialization_interpretation":
                        "offline fixture"
                })

        return pd.DataFrame(rows)

    for region,cfg in REGIONS.items():
        for date in cfg["imerg_dates"]:
            try:
                rec=corrected_imerg_one(
                    region,
                    date,
                    cfg["bbox"]
                )
            except Exception as e:
                rec={
                    "region":region,
                    "date":date,
                    "mode":"live",
                    "status":"FAILED",
                    "error":repr(e)
                }

            rows.append(rec)
            print(
                "IMERG v8.1",
                region,
                date,
                rec["status"]
            )

    return pd.DataFrame(rows)

imerg_v81=run_corrected_imerg()
display(imerg_v81)

# Part V — Corrected CopDEM geospatial resampling

The source patch is now resampled using **rasterio.warp.reproject** with explicit affine transforms.

For each ROI:

1. read the native GLO-30 DSM patch;
2. construct a destination grid with exactly 3× the width and 3× the height and exactly one-third the source pixel spacing;
3. bilinearly reproject onto that aligned fine grid;
4. bilinearly reproject the fine grid back onto the exact original native grid;
5. calculate round-trip errors on a core area that excludes a 3-pixel native border.

This removes the array-coordinate misalignment that produced implausible v8 edge maxima.

In [ ]:
def copdem_tile_name(lat,lon):
    lat_floor=math.floor(lat)
    lon_floor=math.floor(lon)

    ns="N" if lat_floor >= 0 else "S"
    ew="E" if lon_floor >= 0 else "W"

    return (
        "Copernicus_DSM_COG_10_"
        f"{ns}{abs(lat_floor):02d}_00_"
        f"{ew}{abs(lon_floor):03d}_00_DEM"
    )

def geodesic_pixel_size_m(transform,lat,lon):
    from pyproj import Geod

    geod=Geod(ellps="WGS84")

    dx_deg=abs(float(transform.a))
    dy_deg=abs(float(transform.e))

    _,_,dx=geod.inv(
        lon,
        lat,
        lon+dx_deg,
        lat
    )

    _,_,dy=geod.inv(
        lon,
        lat,
        lon,
        lat+dy_deg
    )

    return abs(float(dx)),abs(float(dy))

def corrected_copdem_one(region,bbox,tile_set):
    import rasterio
    from rasterio.warp import transform, reproject, Resampling
    from rasterio.windows import Window
    from affine import Affine

    lon=(bbox[0]+bbox[2])/2
    lat=(bbox[1]+bbox[3])/2

    tile=copdem_tile_name(lat,lon)

    if tile not in tile_set:
        raise RuntimeError(
            f"Expected tile not found: {tile}"
        )

    base=SOURCE_REGISTRY["copdem"]["base"]
    url=f"{base}/{tile}/{tile}.tif"

    env={
        "GDAL_DISABLE_READDIR_ON_OPEN":"EMPTY_DIR",
        "CPL_VSIL_CURL_ALLOWED_EXTENSIONS":".tif,.TIF",
        "GDAL_HTTP_MULTIRANGE":"YES",
        "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES":"YES"
    }

    with rasterio.Env(**env), rasterio.open(url) as src:
        xs,ys=transform(
            "EPSG:4326",
            src.crs,
            [lon],
            [lat]
        )

        row,col=src.index(
            xs[0],
            ys[0]
        )

        size=256

        row0=max(
            0,
            min(src.height-size,row-size//2)
        )

        col0=max(
            0,
            min(src.width-size,col-size//2)
        )

        win=Window(
            col0,
            row0,
            min(size,src.width-col0),
            min(size,src.height-row0)
        )

        src_arr=src.read(
            1,
            window=win,
            masked=True
        ).astype(np.float32)

        native_transform=src.window_transform(win)
        src_crs=src.crs
        src_nodata=src.nodata
        crs=str(src.crs)

    native=src_arr.filled(np.nan)
    source_valid=np.isfinite(native)

    if source_valid.sum() < 100:
        raise RuntimeError(
            "Too few valid CopDEM pixels."
        )

    h,w=native.shape
    factor=3

    fine_h=h*factor
    fine_w=w*factor

    fine_transform=(
        native_transform *
        Affine.scale(
            w/fine_w,
            h/fine_h
        )
    )

    fine=np.full(
        (fine_h,fine_w),
        np.nan,
        dtype=np.float32
    )

    reproject(
        source=native,
        destination=fine,
        src_transform=native_transform,
        src_crs=src_crs,
        src_nodata=np.nan,
        dst_transform=fine_transform,
        dst_crs=src_crs,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear,
        init_dest_nodata=True
    )

    back=np.full(
        native.shape,
        np.nan,
        dtype=np.float32
    )

    reproject(
        source=fine,
        destination=back,
        src_transform=fine_transform,
        src_crs=src_crs,
        src_nodata=np.nan,
        dst_transform=native_transform,
        dst_crs=src_crs,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear,
        init_dest_nodata=True
    )

    valid=(
        np.isfinite(native) &
        np.isfinite(back)
    )

    # Remove edge sensitivity from the round-trip metric.
    core=np.zeros_like(valid,dtype=bool)
    border=3

    if h > 2*border and w > 2*border:
        core[
            border:-border,
            border:-border
        ]=True
    else:
        core[:]=True

    eval_mask=valid & core

    if eval_mask.sum() < 100:
        raise RuntimeError(
            "Too few valid core round-trip pixels."
        )

    err=np.abs(
        back[eval_mask] -
        native[eval_mask]
    )

    native_range=(
        float(np.nanmax(native[eval_mask])) -
        float(np.nanmin(native[eval_mask]))
    )

    dx_m,dy_m=geodesic_pixel_size_m(
        native_transform,
        lat,
        lon
    )

    pixel_inflation=(
        fine.size /
        native.size
    )

    derived_fraction=(
        1 -
        native.size/fine.size
    )

    return {
        "region":region,
        "mode":"live",
        "status":"SUCCESS",
        "tile":tile,
        "crs":crs,
        "surface_model":"DSM",
        "vertical_datum":"EGM2008 / EPSG:3855",
        "native_valid_pixels":
            int(source_valid.sum()),
        "native_core_eval_pixels":
            int(eval_mask.sum()),
        "native_elevation_median_m":
            float(np.nanmedian(native)),
        "native_elevation_min_m":
            float(np.nanmin(native)),
        "native_elevation_max_m":
            float(np.nanmax(native)),
        "native_pixel_width_m_at_roi":
            dx_m,
        "native_pixel_height_m_at_roi":
            dy_m,
        "fine_pixel_width_m_at_roi":
            dx_m/factor,
        "fine_pixel_height_m_at_roi":
            dy_m/factor,
        "upsampling_axis_factor":
            factor,
        "pixel_count_inflation_factor":
            float(pixel_inflation),
        "derived_output_fraction_without_distinct_source_sample":
            float(derived_fraction),
        "roundtrip_core_mae_m":
            float(np.mean(err)),
        "roundtrip_core_median_abs_error_m":
            float(np.median(err)),
        "roundtrip_core_p95_abs_error_m":
            float(np.percentile(err,95)),
        "roundtrip_core_max_abs_error_m":
            float(np.max(err)),
        "roundtrip_mae_fraction_of_native_range":
            float(np.mean(err)/native_range)
            if native_range>0 else np.nan,
        "resampling_method":
            "rasterio.reproject bilinear; exact aligned affine grids"
    }

def run_corrected_copdem():
    rows=[]

    if not LIVE_DATA:
        import rasterio
        from rasterio.warp import reproject, Resampling
        from affine import Affine

        rng=np.random.default_rng(SEED)

        for i,(region,cfg) in enumerate(REGIONS.items()):
            h=w=128
            y,x=np.mgrid[:h,:w]

            native=(
                100 +
                0.4*x +
                0.2*y +
                8*np.sin(x/17) +
                5*np.cos(y/13) +
                rng.normal(0,0.2,(h,w))
            ).astype(np.float32)

            native_transform=Affine(
                0.0002777778,0,cfg["bbox"][0],
                0,-0.0002777778,cfg["bbox"][3]
            )

            factor=3
            fine_transform=(
                native_transform *
                Affine.scale(1/factor,1/factor)
            )

            fine=np.full(
                (h*factor,w*factor),
                np.nan,
                dtype=np.float32
            )

            reproject(
                native,
                fine,
                src_transform=native_transform,
                src_crs="EPSG:4326",
                dst_transform=fine_transform,
                dst_crs="EPSG:4326",
                resampling=Resampling.bilinear
            )

            back=np.full_like(
                native,
                np.nan
            )

            reproject(
                fine,
                back,
                src_transform=fine_transform,
                src_crs="EPSG:4326",
                dst_transform=native_transform,
                dst_crs="EPSG:4326",
                resampling=Resampling.bilinear
            )

            core=np.zeros_like(native,dtype=bool)
            core[3:-3,3:-3]=True

            err=np.abs(
                back[core] -
                native[core]
            )

            rows.append({
                "region":region,
                "mode":"offline_fixture",
                "status":"SUCCESS",
                "surface_model":"DSM",
                "upsampling_axis_factor":3,
                "pixel_count_inflation_factor":9.0,
                "derived_output_fraction_without_distinct_source_sample":
                    8/9,
                "roundtrip_core_mae_m":
                    float(np.mean(err)),
                "roundtrip_core_median_abs_error_m":
                    float(np.median(err)),
                "roundtrip_core_p95_abs_error_m":
                    float(np.percentile(err,95)),
                "roundtrip_core_max_abs_error_m":
                    float(np.max(err)),
                "roundtrip_mae_fraction_of_native_range":
                    float(np.mean(err)/(native.max()-native.min())),
                "resampling_method":
                    "offline aligned affine fixture"
            })

        return pd.DataFrame(rows)

    import requests

    base=SOURCE_REGISTRY["copdem"]["base"]

    r=requests.get(
        base+"/tileList.txt",
        timeout=90
    )

    r.raise_for_status()

    tile_set=set(
        x.strip()
        for x in r.text.splitlines()
        if x.strip()
    )

    for region,cfg in REGIONS.items():
        try:
            rec=corrected_copdem_one(
                region,
                cfg["bbox"],
                tile_set
            )
        except Exception as e:
            rec={
                "region":region,
                "mode":"live",
                "status":"FAILED",
                "error":repr(e)
            }

        rows.append(rec)

        print(
            "CopDEM v8.1",
            region,
            rec["status"]
        )

    return pd.DataFrame(rows)

copdem_v81=run_corrected_copdem()
display(copdem_v81)

# Part VI — Before/after correction audit

This table makes the corrections explicit rather than silently replacing old values.

In [ ]:
imerg_compare = (
    imerg_v8_old[[
        "region",
        "date",
        "status",
        "mean_intervals_per_cell",
        "median_raw_sum_overstatement_factor",
        "median_mean_as_accum_fraction_of_true"
    ]]
    .rename(columns={
        "status":"v8_status",
        "mean_intervals_per_cell":
            "v8_rows_per_cell_misinterpreted_as_completeness",
        "median_raw_sum_overstatement_factor":
            "v8_raw_sum_factor",
        "median_mean_as_accum_fraction_of_true":
            "v8_mean_fraction"
    })
    .merge(
        imerg_v81[[
            "region",
            "date",
            "status",
            "source_time_axis_count",
            "source_time_axis_complete",
            "returned_nonzero_or_unmasked_records",
            "zero_reconstructed_fraction",
            "median_raw_sum_overstatement_factor",
            "median_mean_as_accum_fraction_of_true"
        ]].rename(columns={
            "status":"v81_status",
            "median_raw_sum_overstatement_factor":
                "v81_raw_sum_factor",
            "median_mean_as_accum_fraction_of_true":
                "v81_mean_fraction"
        }),
        on=["region","date"],
        how="outer"
    )
)

copdem_compare = (
    copdem_v8_old[[
        "region",
        "roundtrip_mean_absolute_error_m",
        "roundtrip_max_absolute_error_m"
    ]]
    .rename(columns={
        "roundtrip_mean_absolute_error_m":
            "v8_zoom_roundtrip_mae_m",
        "roundtrip_max_absolute_error_m":
            "v8_zoom_roundtrip_max_m"
    })
    .merge(
        copdem_v81[[
            "region",
            "roundtrip_core_mae_m",
            "roundtrip_core_p95_abs_error_m",
            "roundtrip_core_max_abs_error_m",
            "roundtrip_mae_fraction_of_native_range"
        ]],
        on="region",
        how="outer"
    )
)

print("IMERG correction audit")
display(imerg_compare)

print("CopDEM correction audit")
display(copdem_compare)

# Part VII — Region-cluster bootstrap

Sentinel-2 and IMERG contain repeated dates/seasons within regions.

To avoid treating repeated observations from the same geographic region as fully independent, the publication summary first aggregates each experiment to one effect value per region and then bootstraps **regions**.

The unit of resampling is therefore the geographic region (`n = 4`), not every date/season row.

In [ ]:
def percentile_bootstrap(
    values,
    statistic=np.median,
    B=BOOTSTRAP_B,
    seed=SEED
):
    x=np.asarray(values,dtype=float)
    x=x[np.isfinite(x)]

    if len(x)==0:
        return {
            "n":0,
            "point":np.nan,
            "ci95_low":np.nan,
            "ci95_high":np.nan
        }

    rng=np.random.default_rng(seed)
    point=float(statistic(x))

    sims=np.empty(B,dtype=float)

    for b in range(B):
        s=rng.choice(
            x,
            size=len(x),
            replace=True
        )
        sims[b]=statistic(s)

    lo,hi=np.percentile(
        sims,
        [2.5,97.5]
    )

    return {
        "n":int(len(x)),
        "point":point,
        "ci95_low":float(lo),
        "ci95_high":float(hi)
    }

# Sentinel: median across seasons within each region.
sentinel_success=sentinel_v8[
    sentinel_v8.status.eq("SUCCESS")
].copy()

sentinel_region=(
    sentinel_success
    .groupby("region")
    .agg(
        spectral_mae=(
            "mean_absolute_index_difference",
            "median"
        ),
        sign_flip_fraction=(
            "sign_flip_fraction",
            "median"
        )
    )
    .reset_index()
)

# WorldCover: already one record per region.
worldcover_region=worldcover_v8[
    worldcover_v8.status.eq("SUCCESS")
][[
    "region",
    "bilinear_invalid_fraction",
    "nearest_valid_fraction"
]].copy()

# IMERG: median across dates within region, use only successful rainy results.
imerg_success=imerg_v81[
    imerg_v81.status.eq("SUCCESS")
].copy()

imerg_region=(
    imerg_success
    .groupby("region")
    .agg(
        raw_sum_factor=(
            "median_raw_sum_overstatement_factor",
            "median"
        ),
        mean_as_accum_fraction=(
            "median_mean_as_accum_fraction_of_true",
            "median"
        ),
        zero_reconstructed_fraction=(
            "zero_reconstructed_fraction",
            "median"
        )
    )
    .reset_index()
)

copdem_region=copdem_v81[
    copdem_v81.status.eq("SUCCESS")
][[
    "region",
    "derived_output_fraction_without_distinct_source_sample",
    "pixel_count_inflation_factor",
    "roundtrip_core_mae_m",
    "roundtrip_core_p95_abs_error_m",
    "roundtrip_mae_fraction_of_native_range"
]].copy()

bootstrap_specs = [
    (
        "Sentinel spectral-role mismatch",
        sentinel_region,
        "spectral_mae",
        "Mean absolute NDVI-like index difference"
    ),
    (
        "Sentinel spectral sign flips",
        sentinel_region,
        "sign_flip_fraction",
        "Sign-flip fraction"
    ),
    (
        "WorldCover bilinear categorical corruption",
        worldcover_region,
        "bilinear_invalid_fraction",
        "Invalid categorical-code fraction"
    ),
    (
        "IMERG raw rate sum mislabeled accumulation",
        imerg_region,
        "raw_sum_factor",
        "Numeric overstatement factor"
    ),
    (
        "IMERG full-day mean rate mislabeled accumulation",
        imerg_region,
        "mean_as_accum_fraction",
        "Fraction of true daily accumulation"
    ),
    (
        "CopDEM 30m-to-fine-grid unsupported information claim",
        copdem_region,
        "derived_output_fraction_without_distinct_source_sample",
        "Fine-grid fraction without distinct source samples"
    ),
    (
        "CopDEM aligned round-trip interpolation",
        copdem_region,
        "roundtrip_core_p95_abs_error_m",
        "Core p95 round-trip absolute error (m)"
    )
]

bootstrap_rows=[]

for i,(exp,df,col,metric) in enumerate(bootstrap_specs):
    stats=percentile_bootstrap(
        pd.to_numeric(
            df[col],
            errors="coerce"
        ),
        statistic=np.median,
        seed=SEED+i
    )

    bootstrap_rows.append({
        "experiment":exp,
        "metric":metric,
        "region_n":stats["n"],
        "median":stats["point"],
        "ci95_low":stats["ci95_low"],
        "ci95_high":stats["ci95_high"]
    })

bootstrap_df=pd.DataFrame(bootstrap_rows)
display(bootstrap_df)

# Part VIII — Corrected publication summary table

In [ ]:
publication_rows=[]

def bootrow(name):
    return bootstrap_df[
        bootstrap_df.experiment.eq(name)
    ].iloc[0]

r=bootrow("Sentinel spectral-role mismatch")
r2=bootrow("Sentinel spectral sign flips")

publication_rows.append({
    "experiment":"Sentinel-2 spectral-role mismatch",
    "source":"Sentinel-2 L2A",
    "regions":int(r.region_n),
    "live_replicates":int(len(sentinel_success)),
    "primary_effect":
        float(r["median"]),
    "primary_ci95_low":
        float(r["ci95_low"]),
    "primary_ci95_high":
        float(r["ci95_high"]),
    "primary_metric":
        "region-cluster median mean absolute index difference",
    "secondary_effect":
        float(r2["median"]),
    "secondary_metric":
        "region-cluster median sign-flip fraction",
    "publication_interpretation":
        "Replacing required NIR B08 with SWIR B11 materially changes an NDVI-labelled product despite successful raster computation."
})

r=bootrow("WorldCover bilinear categorical corruption")

publication_rows.append({
    "experiment":"WorldCover categorical interpolation",
    "source":"ESA WorldCover 10 m 2021",
    "regions":int(r.region_n),
    "live_replicates":int(len(worldcover_region)),
    "primary_effect":
        float(r["median"]),
    "primary_ci95_low":
        float(r["ci95_low"]),
    "primary_ci95_high":
        float(r["ci95_high"]),
    "primary_metric":
        "region median invalid categorical-code fraction after bilinear interpolation",
    "secondary_effect":
        float(worldcover_region.nearest_valid_fraction.median()),
    "secondary_metric":
        "nearest-neighbour valid-class fraction",
    "publication_interpretation":
        "Continuous interpolation creates class values absent from the categorical ontology; nearest-neighbour preserves valid class codes."
})

r=bootrow("IMERG raw rate sum mislabeled accumulation")
r2=bootrow("IMERG full-day mean rate mislabeled accumulation")

publication_rows.append({
    "experiment":"IMERG temporal quantity semantics",
    "source":"GPM IMERG Final V07 half-hourly precipitation rate",
    "regions":int(r.region_n),
    "live_replicates":int(
        imerg_success.source_time_axis_complete.fillna(False).sum()
    ),
    "primary_effect":
        float(r["median"]),
    "primary_ci95_low":
        float(r["ci95_low"]),
    "primary_ci95_high":
        float(r["ci95_high"]),
    "primary_metric":
        "raw half-hourly rate-sum / correctly duration-integrated accumulation",
    "secondary_effect":
        float(r2["median"]),
    "secondary_metric":
        "full-day mean-rate numeric / true daily accumulation",
    "publication_interpretation":
        "Ignoring the 0.5 h duration doubles the accumulation numeric value; labeling the 24 h mean rate as accumulation yields only 1/24 of the true daily accumulation for rainy cells."
})

r=bootrow("CopDEM 30m-to-fine-grid unsupported information claim")
r2=bootrow("CopDEM aligned round-trip interpolation")

publication_rows.append({
    "experiment":"CopDEM unsupported spatial-resolution gain",
    "source":"Copernicus DEM GLO-30 DSM",
    "regions":int(r.region_n),
    "live_replicates":int(len(copdem_region)),
    "primary_effect":
        float(r["median"]),
    "primary_ci95_low":
        float(r["ci95_low"]),
    "primary_ci95_high":
        float(r["ci95_high"]),
    "primary_metric":
        "fine-grid fraction without distinct source samples after 3x-per-axis interpolation",
    "secondary_effect":
        float(r2["median"]),
    "secondary_metric":
        "aligned core p95 round-trip interpolation error (m)",
    "publication_interpretation":
        "A 3x-per-axis interpolation creates 9x as many grid cells but no new source observations; numerical smoothness does not justify a native fine-resolution information claim."
})

publication_table=pd.DataFrame(publication_rows)
display(publication_table)

# Part IX — Corrected real-data counterfactual table

The labels below are **scientific rule labels for paired counterfactuals**, not independent human labels.

In [ ]:
counterfactual_rows=[]

# Sentinel
for _,r in sentinel_success.iterrows():
    base={
        "region":r.region,
        "source":"Sentinel-2 L2A",
        "replicate":r.date_range,
        "data_mode":"live"
    }

    counterfactual_rows.extend([
        {
            **base,
            "variant":"valid",
            "operation":"NDVI using B08 NIR and B04 red",
            "rule_label":"PROVEN",
            "effect_metric":"mean_absolute_difference_vs_invalid_variant",
            "effect_value":
                r.mean_absolute_index_difference
        },
        {
            **base,
            "variant":"invalid",
            "operation":"Substitute B11 SWIR for B08 while labeling output NDVI",
            "rule_label":"REFUTED",
            "effect_metric":"sign_flip_fraction",
            "effect_value":
                r.sign_flip_fraction
        }
    ])

# WorldCover
for _,r in worldcover_region.iterrows():
    base={
        "region":r.region,
        "source":"ESA WorldCover",
        "replicate":"2021",
        "data_mode":"live"
    }

    counterfactual_rows.extend([
        {
            **base,
            "variant":"valid",
            "operation":"Nearest-neighbour categorical resampling",
            "rule_label":"PROVEN",
            "effect_metric":"valid_class_fraction",
            "effect_value":
                r.nearest_valid_fraction
        },
        {
            **base,
            "variant":"invalid",
            "operation":"Bilinear interpolation of nominal class codes",
            "rule_label":"REFUTED",
            "effect_metric":"invalid_class_fraction",
            "effect_value":
                r.bilinear_invalid_fraction
        }
    ])

# IMERG
for _,r in imerg_success.iterrows():
    base={
        "region":r.region,
        "source":"GPM IMERG Final V07",
        "replicate":r.date,
        "data_mode":r["mode"]
    }

    counterfactual_rows.extend([
        {
            **base,
            "variant":"valid",
            "operation":"Integrate rate over each 0.5 h interval",
            "rule_label":"PROVEN",
            "effect_metric":"mean_daily_accumulation_mm",
            "effect_value":
                r.mean_correct_daily_accum_mm
        },
        {
            **base,
            "variant":"invalid",
            "operation":"Sum MM/Hr values without multiplying by 0.5 h and label result accumulation",
            "rule_label":"REFUTED",
            "effect_metric":"numeric_overstatement_factor",
            "effect_value":
                r.median_raw_sum_overstatement_factor
        },
        {
            **base,
            "variant":"invalid",
            "operation":"Take full-day mean MM/Hr and label numeric value daily accumulation",
            "rule_label":"REFUTED",
            "effect_metric":"fraction_of_true_accumulation",
            "effect_value":
                r.median_mean_as_accum_fraction_of_true
        }
    ])

# CopDEM
for _,r in copdem_region.iterrows():
    base={
        "region":r.region,
        "source":"Copernicus DEM GLO-30",
        "replicate":"DSM patch",
        "data_mode":"live" if LIVE_DATA else "offline_fixture"
    }

    counterfactual_rows.extend([
        {
            **base,
            "variant":"valid",
            "operation":"Interpolate onto finer grid while retaining GLO-30 source provenance",
            "rule_label":"PROVEN",
            "effect_metric":"pixel_count_inflation_factor",
            "effect_value":
                r.pixel_count_inflation_factor
        },
        {
            **base,
            "variant":"invalid",
            "operation":"Claim the interpolated fine grid contains native fine-resolution source information",
            "rule_label":"REFUTED",
            "effect_metric":"fraction_without_distinct_source_samples",
            "effect_value":
                r.derived_output_fraction_without_distinct_source_sample
        }
    ])

counterfactual_v81=pd.DataFrame(
    counterfactual_rows
)

display(counterfactual_v81.head(20))
print("Rows:",len(counterfactual_v81))

# Part X — Robustness and publication gates

The final table is marked publication-ready only if:

- all 8 IMERG date queries expose 48 source time coordinates;
- both IMERG counterfactual ratios agree numerically with the physical-duration identities on rainy cells;
- all 4 CopDEM regions complete the aligned reprojection round trip;
- the corrected CopDEM maximum/core errors are finite;
- no offline fixtures are mixed into a live publication run.

In [ ]:
imerg_success_rows=imerg_v81[
    imerg_v81.status.eq("SUCCESS")
].copy()

imerg_rainy=imerg_success_rows[
    pd.to_numeric(
        imerg_success_rows.rainy_grid_cells,
        errors="coerce"
    ).fillna(0) > 0
].copy()

raw_factor_ok = (
    np.allclose(
        pd.to_numeric(
            imerg_rainy.median_raw_sum_overstatement_factor,
            errors="coerce"
        ),
        2.0,
        rtol=1e-6,
        atol=1e-8,
        equal_nan=False
    )
    if len(imerg_rainy)
    else False
)

mean_factor_ok = (
    np.allclose(
        pd.to_numeric(
            imerg_rainy.median_mean_as_accum_fraction_of_true,
            errors="coerce"
        ),
        1/24,
        rtol=1e-6,
        atol=1e-8,
        equal_nan=False
    )
    if len(imerg_rainy)
    else False
)

publication_gates={
    "all_8_imerg_dates_success":
        len(imerg_success_rows)==8,

    "all_imerg_time_axes_have_48_half_hours":
        (
            len(imerg_success_rows)==8 and
            imerg_success_rows.source_time_axis_complete.fillna(False).all()
        ),

    "imerg_raw_sum_factor_matches_physics":
        raw_factor_ok,

    "imerg_mean_fraction_matches_physics":
        mean_factor_ok,

    "all_4_copdem_regions_success":
        int((copdem_v81.status=="SUCCESS").sum())==4,

    "copdem_corrected_errors_finite":
        (
            copdem_region.roundtrip_core_p95_abs_error_m
            .apply(np.isfinite)
            .all()
            if len(copdem_region)==4
            else False
        ),

    "sentinel_v8_all_live_success":
        (
            len(sentinel_success)==8 and
            set(sentinel_success["mode"])=={"live"}
        ),

    "worldcover_v8_all_live_success":
        (
            len(worldcover_region)==4 and
            set(
                worldcover_v8[
                    worldcover_v8.status.eq("SUCCESS")
                ]["mode"]
            )=={"live"}
        ),

    "no_offline_fixture_in_publication_run":
        (
            not LIVE_DATA or
            (
                set(imerg_success_rows["mode"])=={"live"} and
                set(copdem_v81[copdem_v81.status.eq("SUCCESS")]["mode"])=={"live"}
            )
        )
}

gates_df=pd.DataFrame([
    {"gate":k,"passed":bool(v)}
    for k,v in publication_gates.items()
])

display(gates_df)

PUBLICATION_READY=bool(
    gates_df.passed.all()
)

print("PUBLICATION_READY =",PUBLICATION_READY)

# Part XI — Candidate plots and one final stitched composite figure

Each panel is first generated as its own independent plot. The four PNGs are then stitched into one 2×2 image for the journal's one-figure limit.

Panels:

**A.** Sentinel spectral-role index difference  
**B.** WorldCover invalid class-code fraction  
**C.** IMERG missing-duration overstatement factor  
**D.** CopDEM corrected core p95 round-trip error, annotated in the table with the 9× grid inflation / 88.89% derived-grid fraction

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw

FIGURES=[]

# Region display labels
pretty = {
    "Bengaluru_India":"Bengaluru",
    "Rotterdam_Netherlands":"Rotterdam",
    "CentralValley_USA":"Central Valley",
    "Manaus_Brazil":"Manaus"
}

# A — Sentinel
panel_a=WORK/"panel_A_sentinel.png"

plt.figure(figsize=(7,4.5))
x=sentinel_region.copy()
x["label"]=x.region.map(pretty).fillna(x.region)

plt.bar(
    x.label,
    x.spectral_mae
)

plt.ylabel(
    "Median mean absolute index difference"
)
plt.xlabel("Region")
plt.xticks(rotation=20,ha="right")
plt.tight_layout()
plt.savefig(panel_a,dpi=220)
plt.close()
FIGURES.append(panel_a)

# B — WorldCover
panel_b=WORK/"panel_B_worldcover.png"

plt.figure(figsize=(7,4.5))
x=worldcover_region.copy()
x["label"]=x.region.map(pretty).fillna(x.region)

plt.bar(
    x.label,
    x.bilinear_invalid_fraction
)

plt.ylabel(
    "Bilinear invalid class-code fraction"
)
plt.xlabel("Region")
plt.ylim(0, max(0.25,float(x.bilinear_invalid_fraction.max())*1.2))
plt.xticks(rotation=20,ha="right")
plt.tight_layout()
plt.savefig(panel_b,dpi=220)
plt.close()
FIGURES.append(panel_b)

# C — IMERG
panel_c=WORK/"panel_C_imerg.png"

plt.figure(figsize=(7,4.5))
x=imerg_region.copy()
x["label"]=x.region.map(pretty).fillna(x.region)

plt.bar(
    x.label,
    x.raw_sum_factor
)

plt.axhline(
    2.0,
    linestyle="--",
    linewidth=1
)

plt.ylabel(
    "Raw rate-sum / correct accumulation"
)
plt.xlabel("Region")
plt.ylim(0,2.4)
plt.xticks(rotation=20,ha="right")
plt.tight_layout()
plt.savefig(panel_c,dpi=220)
plt.close()
FIGURES.append(panel_c)

# D — CopDEM
panel_d=WORK/"panel_D_copdem.png"

plt.figure(figsize=(7,4.5))
x=copdem_region.copy()
x["label"]=x.region.map(pretty).fillna(x.region)

plt.bar(
    x.label,
    x.roundtrip_core_p95_abs_error_m
)

plt.ylabel(
    "Aligned core p95 round-trip error (m)"
)
plt.xlabel("Region")
plt.xticks(rotation=20,ha="right")
plt.tight_layout()
plt.savefig(panel_d,dpi=220)
plt.close()
FIGURES.append(panel_d)

# Stitch without matplotlib subplots.
loaded=[Image.open(p).convert("RGB") for p in FIGURES]

max_w=max(im.width for im in loaded)
max_h=max(im.height for im in loaded)

pads=[]

for idx,im in enumerate(loaded):
    canvas=Image.new(
        "RGB",
        (max_w,max_h),
        "white"
    )

    canvas.paste(
        im,
        (
            (max_w-im.width)//2,
            (max_h-im.height)//2
        )
    )

    draw=ImageDraw.Draw(canvas)

    draw.text(
        (18,14),
        chr(ord("A")+idx),
        fill="black"
    )

    pads.append(canvas)

composite=Image.new(
    "RGB",
    (2*max_w,2*max_h),
    "white"
)

composite.paste(pads[0],(0,0))
composite.paste(pads[1],(max_w,0))
composite.paste(pads[2],(0,max_h))
composite.paste(pads[3],(max_w,max_h))

composite_path=WORK/"FINAL_COMPOSITE_FIGURE_v8_1.png"
composite.save(
    composite_path,
    quality=95
)

print("Composite:",composite_path)

# Part XII — Manuscript-ready results text

The text is generated only from corrected statistics.

If any publication gate fails, it is explicitly marked as provisional.

In [ ]:
def pct(x):
    return f"{100*x:.2f}%"

sent_pub=publication_table[
    publication_table.experiment.eq(
        "Sentinel-2 spectral-role mismatch"
    )
].iloc[0]

wc_pub=publication_table[
    publication_table.experiment.eq(
        "WorldCover categorical interpolation"
    )
].iloc[0]

im_pub=publication_table[
    publication_table.experiment.eq(
        "IMERG temporal quantity semantics"
    )
].iloc[0]

dem_pub=publication_table[
    publication_table.experiment.eq(
        "CopDEM unsupported spatial-resolution gain"
    )
].iloc[0]

status_line = (
    "PUBLICATION-READY under v8.1 correction gates."
    if PUBLICATION_READY
    else
    "PROVISIONAL: one or more v8.1 publication gates failed; inspect the gate table before using these numbers."
)

results_text=f"""
SciGeoGuard-X v8.1 real-data counterfactual replication
=======================================================

Status:
{status_line}

Sentinel-2:
Across four regions and two seasonal windows per region, replacing the NIR B08 role with SWIR B11 in an NDVI-labelled computation produced a region-cluster median mean absolute index difference of {sent_pub.primary_effect:.4f} (95% bootstrap CI {sent_pub.primary_ci95_low:.4f}–{sent_pub.primary_ci95_high:.4f}). The corresponding region-cluster median sign-flip fraction was {pct(sent_pub.secondary_effect)}.

ESA WorldCover:
Bilinear interpolation of nominal WorldCover codes produced a median invalid-class fraction of {pct(wc_pub.primary_effect)} across four regions (95% bootstrap CI {pct(wc_pub.primary_ci95_low)}–{pct(wc_pub.primary_ci95_high)}), whereas nearest-neighbour resampling retained a median valid-class fraction of {pct(wc_pub.secondary_effect)}.

GPM IMERG V07:
After verifying the 48-point half-hourly source time axis and reconstructing zero-rain slots under the ERDDAP 0.0 missing/fill serialization convention, summing rate samples without the required 0.5 h duration yielded a median numeric overstatement factor of {im_pub.primary_effect:.4f}×. Treating the full-day mean precipitation rate as daily accumulation yielded {im_pub.secondary_effect:.6f} of the true accumulation for rainy cells (the theoretical value is 1/24 = {1/24:.6f}).

Copernicus DEM:
Threefold interpolation along each raster axis increased grid-cell count ninefold, so {pct(dem_pub.primary_effect)} of fine-grid positions did not correspond to distinct source samples. With geospatially aligned rasterio reprojection and a core-area comparison, the region-median p95 native→fine→native round-trip interpolation error was {dem_pub.secondary_effect:.4f} m. This numerical smoothness does not constitute new native fine-resolution source information.

Interpretation:
These results are paired real-input counterfactual stress tests. They demonstrate measurable consequences of scientifically invalid but computationally executable processing choices. They are not independent human ground truth.
""".strip()

print(results_text)

(WORK/"MANUSCRIPT_READY_RESULTS_v8_1.txt").write_text(
    results_text,
    encoding="utf-8"
)

# Part XIII — Provenance and correction manifest

In [ ]:
correction_manifest={
    "name":"SciGeoGuard-X Robustness Corrections v8.1",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "live_data":LIVE_DATA,
    "source_v8_preserved":{
        "Sentinel2_multiregion_results":
            "embedded from user's completed live v8 run",
        "WorldCover_multiregion_results":
            "embedded from user's completed live v8 run"
    },
    "corrections":{
        "IMERG":{
            "old_issue":
                "precipitation CSV row count was interpreted as temporal completeness",
            "new_method":
                "verify 48 half-hour source time coordinates separately; reconstruct absent 0.0-serialized precipitation slots as zero rain; calculate full-day dimensional counterfactuals"
        },
        "CopDEM":{
            "old_issue":
                "scipy.ndimage.zoom array-coordinate round trip produced edge/alignment artifacts",
            "new_method":
                "rasterio.reproject with exact aligned affine source/fine/native transforms and core-area round-trip metrics"
        }
    },
    "bootstrap":{
        "unit":"geographic region",
        "regions":4,
        "replicates":BOOTSTRAP_B,
        "statistic":"median",
        "interval":"percentile 95%"
    },
    "publication_ready":PUBLICATION_READY,
    "source_registry":SOURCE_REGISTRY
}

(WORK/"CORRECTION_AND_PROVENANCE_MANIFEST.json").write_text(
    json.dumps(
        correction_manifest,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        correction_manifest,
        indent=2,
        default=str
    )[:3500]
)

# Part XIV — Export the corrected publication bundle

In [ ]:
# Preserve successful v8 source tables.
sentinel_v8.to_csv(
    WORK/"Sentinel2_v8_preserved_live_results.csv",
    index=False
)

worldcover_v8.to_csv(
    WORK/"WorldCover_v8_preserved_live_results.csv",
    index=False
)

# Corrected results.
imerg_v81.to_csv(
    WORK/"IMERG_v8_1_corrected_results.csv",
    index=False
)

copdem_v81.to_csv(
    WORK/"CopDEM_v8_1_corrected_results.csv",
    index=False
)

imerg_compare.to_csv(
    WORK/"IMERG_v8_vs_v8_1_correction_audit.csv",
    index=False
)

copdem_compare.to_csv(
    WORK/"CopDEM_v8_vs_v8_1_correction_audit.csv",
    index=False
)

sentinel_region.to_csv(
    WORK/"Sentinel_region_level_effects.csv",
    index=False
)

worldcover_region.to_csv(
    WORK/"WorldCover_region_level_effects.csv",
    index=False
)

imerg_region.to_csv(
    WORK/"IMERG_region_level_effects.csv",
    index=False
)

copdem_region.to_csv(
    WORK/"CopDEM_region_level_effects.csv",
    index=False
)

bootstrap_df.to_csv(
    WORK/"Region_cluster_bootstrap_95CI.csv",
    index=False
)

publication_table.to_csv(
    WORK/"FINAL_PUBLICATION_RESULTS_TABLE_v8_1.csv",
    index=False
)

counterfactual_v81.to_csv(
    WORK/"FINAL_REALDATA_COUNTERFACTUALS_v8_1.csv",
    index=False
)

gates_df.to_csv(
    WORK/"PUBLICATION_GATES_v8_1.csv",
    index=False
)

README=f"""
SciGeoGuard-X Robustness Corrections v8.1
==========================================

LIVE_DATA = {LIVE_DATA}
PUBLICATION_READY = {PUBLICATION_READY}

Preserved from completed live v8:
- Sentinel-2 results
- ESA WorldCover results

Corrected in v8.1:
- GPM IMERG V07 temporal semantics
- Copernicus DEM aligned resampling

Statistical synthesis:
- region-level aggregation
- 10,000-sample percentile cluster bootstrap
- geographic region is the bootstrap unit

Primary publication files:
- FINAL_PUBLICATION_RESULTS_TABLE_v8_1.csv
- Region_cluster_bootstrap_95CI.csv
- FINAL_REALDATA_COUNTERFACTUALS_v8_1.csv
- FINAL_COMPOSITE_FIGURE_v8_1.png
- MANUSCRIPT_READY_RESULTS_v8_1.txt
- PUBLICATION_GATES_v8_1.csv
- CORRECTION_AND_PROVENANCE_MANIFEST.json

Interpretation:
Real-data paired counterfactual replication.
Not independent human ground truth.
Human validation remains the v7 stage.
"""

(WORK/"README_v8_1.txt").write_text(
    README,
    encoding="utf-8"
)

# Hash all files except the hash table itself.
hash_rows=[]

for p in sorted(WORK.iterdir()):
    if p.is_file() and p.name!="FILE_HASHES_v8_1.csv":
        hash_rows.append({
            "file":p.name,
            "sha256":
                hashlib.sha256(
                    p.read_bytes()
                ).hexdigest(),
            "bytes":p.stat().st_size
        })

pd.DataFrame(hash_rows).to_csv(
    WORK/"FILE_HASHES_v8_1.csv",
    index=False
)

zip_path=shutil.make_archive(
    str(ROOT/"SciGeoGuard_X_v8_1_Robustness_Corrections"),
    "zip",
    root_dir=WORK
)

print("Created:",zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("ZIP path:",zip_path)

# Final decision rule

Use the v8.1 results in the manuscript **only if**:

```python
PUBLICATION_READY == True
```

If it is `False`, upload the ZIP and inspect `PUBLICATION_GATES_v8_1.csv` before using any corrected values.

Once v8.1 passes, the empirical evidence stack is:

1. **v3** — controlled generated benchmark;
2. **v6** — strong decomposed 7B LLM baselines;
3. **v8.1** — corrected multi-region live-data counterfactual replication;
4. **v7** — independent human validation when reviewers complete their blinded forms.

At that point, additional model experiments are lower priority than manuscript synthesis and independent human review.